# Redes convolucionais (CNNs)

**Objetivo:** ver a **convolução** funcionando — primeiro na mão, com `numpy`, para um filtro de borda acender sobre um dígito; depois montar uma pequena **CNN** em PyTorch e comparar com a rede densa do tópico anterior, nos **mesmos** dígitos.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)
import torch

## 1. Uma imagem para brincar

Pegamos um dígito do `load_digits` (imagens 8×8 em tons de cinza) e o mostramos como matriz de pixels.

In [ ]:
from sklearn.datasets import load_digits

digitos = load_digits()
imagem = digitos.images[0]                 # um "7"? um "0"? (8x8)
print("rotulo:", digitos.target[0], "| shape:", imagem.shape)
px.imshow(imagem, color_continuous_scale="gray_r",
          title="Um digito 8x8").show()

## 2. Convolução na mão

Um **filtro** (kernel) 3×3 desliza pela imagem. Em cada posição, multiplica os 9 pesos pela janelinha de pixels e soma. Este kernel responde a **bordas verticais** — a imagem mudando na horizontal. Note o laço explícito: é **o mesmo filtro** em toda posição.

In [ ]:
# kernel de borda vertical (Sobel-x)
kernel = np.array([[1, 0, -1],
                   [2, 0, -2],
                   [1, 0, -1]], dtype=float)

alt, larg = imagem.shape
mapa = np.zeros((alt - 2, larg - 2))        # mapa de ativacao 6x6
for i in range(alt - 2):
    for j in range(larg - 2):
        janela = imagem[i:i+3, j:j+3]
        mapa[i, j] = np.sum(kernel * janela)

print("mapa de ativacao:", mapa.shape)
px.imshow(np.abs(mapa), color_continuous_scale="gray_r",
          title="Resposta ao filtro de borda vertical (|valor|)").show()

As colunas onde o dígito muda de claro para escuro **acendem** no mapa. Trocar o kernel muda o padrão detectado — uma CNN **aprende** esses filtros sozinha, em vez de recebê-los prontos.

## 3. Uma CNN de verdade em PyTorch

`Conv2d(1→8, 3×3)` aprende **8** filtros; `ReLU` corta os negativos; `MaxPool2d(2)` resume e encolhe; achatamos e uma camada densa classifica os 10 dígitos. Poucos pesos, graças ao compartilhamento.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = digitos.images / 16.0                   # normaliza 0..1
y = digitos.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=SEMENTE, stratify=y)
# PyTorch espera (amostras, canais, altura, largura)
ent_tr = torch.tensor(X_tr, dtype=torch.float32).unsqueeze(1)
ent_te = torch.tensor(X_te, dtype=torch.float32).unsqueeze(1)
alvo_tr = torch.tensor(y_tr, dtype=torch.long)
print("entrada de treino:", tuple(ent_tr.shape), "  (N, canais, 8, 8)")

In [ ]:
torch.manual_seed(SEMENTE)
rede = torch.nn.Sequential(
    torch.nn.Conv2d(1, 8, kernel_size=3),   # 8x8 -> 6x6, 8 filtros
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),                  # 6x6 -> 3x3
    torch.nn.Flatten(),                     # 8*3*3 = 72
    torch.nn.Linear(8 * 3 * 3, 10))
custo_fn = torch.nn.CrossEntropyLoss()
oti = torch.optim.Adam(rede.parameters(), lr=0.01)

perdas = []
for epoca in range(60):
    ordem = torch.randperm(len(ent_tr))
    for i in range(0, len(ent_tr), 64):
        idx = ordem[i:i+64]
        perda = custo_fn(rede(ent_tr[idx]), alvo_tr[idx])
        oti.zero_grad(); perda.backward(); oti.step()
    perdas.append(perda.item())

with torch.no_grad():
    previsto = rede(ent_te).argmax(dim=1).numpy()
print("acuracia da CNN:", round(accuracy_score(y_te, previsto), 3))
print("numero de pesos:", sum(p.numel() for p in rede.parameters()))

In [ ]:
figura = go.Figure(go.Scatter(y=perdas, mode="lines", line=dict(color=VERDE)))
figura.update_layout(title="Perda do treino da CNN (digitos)",
                     xaxis_title="epoca", yaxis_title="CrossEntropy", height=320,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 4. O que mudou em relação à rede densa?

Nos dígitos 8×8 — minúsculos e quase separáveis — a CNN empata ou supera a rede densa por pouco. O ponto **não** é o placar: é que a CNN chega lá **respeitando a estrutura da imagem** e com filtros reaproveitados por toda ela. Em imagens **grandes e cruas** (não 8×8, mas centenas de milhares de pixels) essa vantagem deixa de ser sutil e vira a diferença entre funcionar e não funcionar.

## Exercício

A `Conv2d(1, 8, 3)` tem quantos pesos (sem os vieses)? Compare com uma camada densa que ligasse a imagem 8×8 (64 pixels) a 8 neurônios.

<details><summary>Ver resposta</summary>

A convolucional tem $8 \times (3\times3) = 72$ pesos: 8 filtros de 9 pesos, cada filtro reusado em toda a imagem. A densa teria $64 \times 8 = 512$ pesos — e **cresceria com o tamanho da imagem**, enquanto o filtro convolucional continua com 9 pesos independentemente da resolução. É o **compartilhamento de pesos** que torna a CNN viável em imagens grandes.

</details>